In [1]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [6]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=",")

EmptyDataError: No columns to parse from file

In [11]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=",")

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=";")

In [8]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=";")

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.head()

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifier le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
## On étudie la variable dateCreationEtablissement  => quelle est la dernière date de création 
# max : 2027-02-01 => étrange, pourquoi i) des dates dans le futur, et ii) la date de mise à jour la plus récente est moins récente (cf cellule plus bas) ??
# min : 1900-01-01 (évidement oui, également la date d'entrée en vigueur de code civil allemand fyi)


In [ ]:
##  Garder uniquement les établissements actifs à une date t
### On peut choisir arbitrairement le 1er janvier de l'année des élections municipales comme condition
cond_date = "2026-01-01"
print(cond_date)
fraise = bistrots_sirene.copy()

fraise = fraise[(fraise['dateCreationEtablissement'] <= cond_date)]

# compter le nb qu'il reste => la différence est huge, vérifier l'output voir si les conditions sont les bonnes (ou trop restrictives)
print(bistrots_sirene["siret"].nunique())
print(fraise["siret"].nunique())

In [ ]:
# Proposition code alternatif pour décomptes sans copie
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['dateCreationEtablissement'] <= cond_date)]["siret"].nunique())

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .groupby(["codeCommuneEtablissement", "libelleCommuneEtablissement"])["siret"]
    .nunique()
    .rename("nb_etablissements")
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

In [12]:
t2_muni.head(20)

,Unnamed: 0,Code département,Libellé département,Code commune,Libellé commune,Inscrits,Votants,% Votants,Abstentions,% Abstentions,...,Sexe candidat 5,Nuance liste 5,Libellé abrégé de liste 5,Libellé de liste 5,Voix 5,% Voix/inscrits 5,% Voix/exprimés 5,Elu 5,Sièges au CM 5,Sièges au CC 5
0,0,01,Ain,01160,Ferney-Voltaire,4486,1958,"43,65%",2528,"56,35%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01,Ain,01194,Jassans-Riottier,4387,2469,"56,28%",1918,"43,72%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01,Ain,01249,Miribel,7293,3920,"53,75%",3373,"46,25%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01,Ain,01283,Oyonnax,12283,6777,"55,17%",5506,"44,83%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01,Ain,01322,Reyrieux,3993,2310,"57,85%",1683,"42,15%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,01,Ain,01378,Saint-Maurice-de-Gourdans,2354,1513,"64,27%",841,"35,73%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,01,Ain,01427,Trévoux,4336,2845,"65,61%",1491,"34,39%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,01,Ain,01450,Villieu-Loyes-Mollon,3112,1931,"62,05%",1181,"37,95%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,02,Aisne,02011,Ambleny,939,589,"62,73%",350,"37,27%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,02,Aisne,02063,Bellenglise,301,246,"81,73%",55,"18,27%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
t2_muni_26.describe()

In [ ]:
t2_muni_26.nuance.value_counts(dropna=False)

In [ ]:
# Agrégation par commune (pour l'instant, données par bureau de vote)

In [ ]:
# Nombre de listes par ville
# Deux pistes : pivot ou window function

In [ ]:
# Création d'un indicateur de vote 


# TODO data prep

* Filtrer les bistrots actifs à une date t que l'on déterminera [en cours, besoin de se renseigner sur la façon dont est remplie la variable dateDernierTraitementEtablissement] OK
* Agréger les bistrots Sirene en un compte de bistrots par commune OK
* Traiter les données des municipales pour obtenir un indicateur quanti du vote "simili-Attal" (majo présidentielle, en gros) et du vote RN 
* Sélectionner les variables d'intérêt dans chaque table pour en faire quelque chose de lisible
* Fusionner les tables (partir de la table municipale T1 comme référence)